# Data Cleaning for Classification

This notebook prepares a clean dataset for the classifier.

Steps:
1. Load and standardize source files
2. Keep only label, title, abstract
3. Normalize text and remove invalid rows
4. Remove duplicate/conflicting records
5. Save cleaned CSV


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_DIR = Path("Dataset")
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def standardize_frame(frame: pd.DataFrame, title_col: str, abstract_col: str, default_label: str | None = None) -> pd.DataFrame:
    out = pd.DataFrame()
    out["label"] = frame["label"] if "label" in frame.columns else np.nan
    out["title"] = frame[title_col] if title_col in frame.columns else np.nan
    out["abstract"] = frame[abstract_col] if abstract_col in frame.columns else np.nan
    if default_label is not None:
        out["label"] = default_label
    return out[["label", "title", "abstract"]]

applied_raw = pd.read_csv(DATASET_DIR / "applied_papers_1000.csv")
theory_raw = pd.read_csv(DATASET_DIR / "theory_papers_1000.csv")
survey_raw = pd.read_csv(DATASET_DIR / "survey_papers_901.csv")

applied_df = standardize_frame(applied_raw, title_col="title", abstract_col="abstract")
theory_df = standardize_frame(theory_raw, title_col="title", abstract_col="abstract")
survey_df = standardize_frame(survey_raw, title_col="arxiv_title", abstract_col="arxiv_abstract", default_label="SURVEY")

merged_df = pd.concat([applied_df, theory_df, survey_df], ignore_index=True)
print("Merged shape:", merged_df.shape)
merged_df.head(3)


In [ ]:
def normalize_text(value: object) -> str | float:
    if pd.isna(value):
        return np.nan
    text = re.sub(r"\s+", " ", str(value).replace("\n", " "))
    text = text.strip()
    return text if text else np.nan

cleaned_df = merged_df.copy()
for col in ["label", "title", "abstract"]:
    cleaned_df[col] = cleaned_df[col].apply(normalize_text)

before_missing_drop = len(cleaned_df)
cleaned_df = cleaned_df.dropna(subset=["title", "abstract"]).copy()
after_missing_drop = len(cleaned_df)
cleaned_df["label"] = cleaned_df["label"].fillna("UNKNOWN")

cleaned_df["text_key"] = (
    cleaned_df["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    + " || "
    + cleaned_df["abstract"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

conflict_keys = cleaned_df.groupby("text_key")["label"].nunique()
conflict_keys = conflict_keys[conflict_keys > 1].index
before_conflict_drop = len(cleaned_df)
cleaned_df = cleaned_df[~cleaned_df["text_key"].isin(conflict_keys)].copy()
after_conflict_drop = len(cleaned_df)

before_dedup = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset=["label", "text_key"], keep="first").copy()
after_dedup = len(cleaned_df)

print("Rows dropped for missing title/abstract:", before_missing_drop - after_missing_drop)
print("Rows removed for label conflicts:", before_conflict_drop - after_conflict_drop)
print("Rows removed as duplicates:", before_dedup - after_dedup)
print("Rows after cleaning:", len(cleaned_df))

cleaned_df = cleaned_df[["label", "title", "abstract"]]
cleaned_df.head(3)


In [ ]:
cleaned_output = DATA_DIR / "combined_papers_cleaned.csv"
cleaned_df.to_csv(cleaned_output, index=False)

print(f"Saved cleaned dataset to {cleaned_output}")
print("Final shape:", cleaned_df.shape)
print("Columns:", list(cleaned_df.columns))
print(cleaned_df["label"].value_counts())
